# Crack Segmentation — DeepLabV3+ (ResNet50)
Colab version with DeepLabV3+ architecture.

**Setup**:
1. Open in Colab
2. Runtime → T4 GPU
3. Run all cells

**Target:** 90%+ Crack IoU (vs 70% MAnet baseline)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU"
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install -q segmentation-models-pytorch albumentations timm

In [ ]:
from pathlib import Path
import json
import zipfile
import shutil

WORK_DIR = Path('/content')
DATA_DIR = WORK_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_DIR = Path('/content/drive/MyDrive/HeritagePreservation')

print(f"Extracting datasets...")

def extract_dataset(zip_name, data_dir, drive_dir):
    zip_path = drive_dir / zip_name
    out_name = zip_name.replace('.zip', '')
    out_dir = data_dir / out_name
    if not zip_path.exists():
        print(f'  SKIP {zip_name}')
        return
    if out_dir.exists() and any(out_dir.rglob('*.*')):
        print(f'  {zip_name}: cached')
        return
    size_mb = zip_path.stat().st_size / 1e6
    local_zip = Path(f'/content/_tmp_{out_name}.zip')
    print(f'  {zip_name} ({size_mb:.0f}MB)...')
    shutil.copy2(zip_path, local_zip)
    with zipfile.ZipFile(local_zip, 'r') as zf:
        zf.extractall(data_dir)
    local_zip.unlink()

for z in ['masonry.zip', 'crackforest.zip', 'historical_crack.zip']:
    extract_dataset(z, DATA_DIR, DRIVE_DIR)

CKPT_DIR = WORK_DIR / 'checkpoints'
PLOTS_DIR = WORK_DIR / 'plots'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_CKPT = DRIVE_DIR / 'checkpoints' / 'segmentor_v6_deeplab'
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

def backup_to_drive():
    if (CKPT_DIR / 'best.pth').exists():
        shutil.copy2(CKPT_DIR / 'best.pth', DRIVE_CKPT / 'best.pth')
    if (CKPT_DIR / 'history.json').exists():
        shutil.copy2(CKPT_DIR / 'history.json', DRIVE_CKPT / 'history.json')
    print(f"  Backed up")

print(f"Data: {DATA_DIR}")

In [ ]:
import cv2
import numpy as np
import random
from sklearn.model_selection import train_test_split

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
IMG_DIR_NAMES = {'images', 'img', 'image', 'jpegimages', 'rgb', 'data'}
MASK_DIR_NAMES = {'masks', 'mask', 'labels', 'label', 'annotations', 'gts', 'gt'}

def find_image_mask_pairs(root_dir):
    root = Path(root_dir)
    pairs = []
    seen = set()
    for img_dir in root.rglob('*'):
        if not img_dir.is_dir() or img_dir.name.lower() not in IMG_DIR_NAMES:
            continue
        parent = img_dir.parent
        mask_dir = None
        for mn in MASK_DIR_NAMES:
            if (parent / mn).is_dir():
                mask_dir = parent / mn
                break
        if mask_dir is None:
            continue
        for img_path in sorted(img_dir.glob('*.*')):
            if img_path.suffix.lower() not in IMG_EXTS:
                continue
            if str(img_path) in seen:
                continue
            for ext in ['.png', '.jpg', '.bmp', img_path.suffix]:
                mask_path = mask_dir / f'{img_path.stem}{ext}'
                if mask_path.exists():
                    pairs.append((img_path, mask_path))
                    seen.add(str(img_path))
                    break
    return pairs

all_pairs = []
for dataset_dir in sorted(DATA_DIR.iterdir()):
    if dataset_dir.is_dir():
        pairs = find_image_mask_pairs(dataset_dir)
        all_pairs.extend(pairs)
        print(f"{dataset_dir.name:20s} {len(pairs):4d} pairs")

print(f"Total: {len(all_pairs)} pairs")
random.seed(42)
train_p, temp_p = train_test_split(all_pairs, train_size=0.7, random_state=42)
val_p, test_p = train_test_split(temp_p, train_size=0.5, random_state=42)
print(f"Train: {len(train_p)} | Val: {len(val_p)} | Test: {len(test_p)}")

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

def get_transforms(split, size=384):
    if split == 'train':
        return A.Compose([
            A.Resize(size, size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomBrightnessContrast(p=0.3),
            A.GaussNoise(p=0.1),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(size, size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])

In [ ]:
from torch.utils.data import Dataset, DataLoader

_CLAHE = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

class SegDataset(Dataset):
    def __init__(self, pairs, split='train', size=384, transform=None):
        self.pairs = pairs
        self.transform = transform or get_transforms(split, size)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_p, mask_p = self.pairs[idx]
        img = cv2.imread(str(img_p))
        img = np.zeros((512, 512, 3), dtype=np.uint8) if img is None \
              else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        lab[..., 0] = _CLAHE.apply(lab[..., 0])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        
        mask = cv2.imread(str(mask_p), cv2.IMREAD_GRAYSCALE)
        mask = np.zeros((512, 512), dtype=np.uint8) if mask is None else mask
        if mask.shape[:2] != img.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)
        
        aug = self.transform(image=img, mask=mask)
        return aug['image'], aug['mask'].unsqueeze(0)

def make_loaders(train_pairs, val_pairs, test_pairs, size, batch, eval_batch=4):
    train_ds = SegDataset(train_pairs, 'train', size, get_transforms('train', size))
    val_ds = SegDataset(val_pairs, 'val', size, get_transforms('val', size))
    test_ds = SegDataset(test_pairs, 'test', size, get_transforms('val', size))
    return (
        DataLoader(train_ds, batch_size=batch, shuffle=True, num_workers=2),
        DataLoader(val_ds, batch_size=eval_batch, shuffle=False, num_workers=2),
        DataLoader(test_ds, batch_size=eval_batch, shuffle=False, num_workers=2),
        test_ds
    )

In [ ]:
DEVICE = torch.device('cuda')
import segmentation_models_pytorch as smp

def build_model():
    return smp.DeepLabV3Plus(
        encoder_name='resnet50',
        encoder_weights='imagenet',
        in_channels=3,
        classes=1,
        activation=None,
    )

model = build_model().to(DEVICE)
print(f"Model: DeepLabV3+ (ResNet50)")
print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
tversky = smp.losses.TverskyLoss(mode='binary', alpha=0.3, beta=0.7, from_logits=True)
lovasz = smp.losses.LovaszLoss(mode='binary', per_image=False, from_logits=True)

def loss_fn(pred, target):
    return 0.5 * tversky(pred, target) + 0.5 * lovasz(pred, target)

print("Loss: Tversky + Lovasz")

In [ ]:
from tqdm.notebook import tqdm
from torch.amp import GradScaler, autocast

def compute_metrics(pred_logits, target, threshold=0.5):
    pred_binary = (torch.sigmoid(pred_logits) > threshold).long()
    tp, fp, fn, tn = smp.metrics.get_stats(pred_binary, target.long(), mode='binary')
    return {
        'iou': float(smp.metrics.iou_score(tp, fp, fn, tn, reduction='micro')),
        'dice': float(smp.metrics.f1_score(tp, fp, fn, tn, reduction='micro')),
    }

def run_epoch(model, loader, optimizer, scaler, train):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_iou, all_dice = [], []
    with (torch.enable_grad() if train else torch.no_grad()):
        for imgs, masks in tqdm(loader, leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            with autocast('cuda'):
                pred = model(imgs)
                loss = loss_fn(pred, masks)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * imgs.size(0)
            with torch.no_grad():
                m = compute_metrics(pred, masks)
                all_iou.append(m['iou'])
                all_dice.append(m['dice'])
    return {
        'loss': total_loss / len(loader.dataset),
        'iou': np.mean(all_iou),
        'dice': np.mean(all_dice),
    }

## Phase 1: Encoder Frozen @ 256px

In [ ]:
for p in model.encoder.parameters():
    p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'P1: frozen  |  trainable: {trainable:,}  |  256px  |  20ep  |  lr=5e-4')

train_loader, val_loader, test_loader, _ = make_loaders(train_p, val_p, test_p, 256, 16)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-5)
scaler = GradScaler('cuda')

history = {k: [] for k in ['train_loss', 'train_iou', 'val_loss', 'val_iou']}
best_iou = 0.0
torch.cuda.empty_cache()

for epoch in range(1, 21):
    tr = run_epoch(model, train_loader, optimizer, scaler, True)
    va = run_epoch(model, val_loader, optimizer, scaler, False)
    scheduler.step()
    for k, v in [('train_loss', tr['loss']), ('train_iou', tr['iou']), ('val_loss', va['loss']), ('val_iou', va['iou'])]:
        history[k].append(v)
    if va['iou'] > best_iou:
        best_iou = va['iou']
        torch.save({'model_state': model.state_dict(), 'val_iou': best_iou}, CKPT_DIR / 'best.pth')
        if epoch % 5 == 0:
            print(f'P1 E{epoch:2d}  loss={tr["loss"]:.4f}  val_iou={va["iou"]:.4f} ✓')
    elif epoch % 5 == 0:
        print(f'P1 E{epoch:2d}  loss={tr["loss"]:.4f}  val_iou={va["iou"]:.4f}')

print(f'P1 done. Best: {best_iou:.4f}')
backup_to_drive()

## Phase 2: Unfrozen @ 384px

In [ ]:
for p in model.parameters():
    p.requires_grad = True

ckpt = torch.load(CKPT_DIR / 'best.pth', map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded P1 ({ckpt["val_iou"]:.4f})')
print(f'P2: unfrozen  |  384px  |  70ep  |  lr=1e-4')

train_loader, val_loader, test_loader, _ = make_loaders(train_p, val_p, test_p, 384, 16)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=35, T_mult=2, eta_min=1e-6)
scaler = GradScaler('cuda')

best_iou = 0.0
torch.cuda.empty_cache()

for epoch in range(1, 71):
    tr = run_epoch(model, train_loader, optimizer, scaler, True)
    va = run_epoch(model, val_loader, optimizer, scaler, False)
    scheduler.step()
    for k, v in [('train_loss', tr['loss']), ('train_iou', tr['iou']), ('val_loss', va['loss']), ('val_iou', va['iou'])]:
        history[k].append(v)
    if va['iou'] > best_iou:
        best_iou = va['iou']
        torch.save({'model_state': model.state_dict(), 'val_iou': best_iou}, CKPT_DIR / 'best.pth')
        if epoch % 10 == 0:
            print(f'P2 E{epoch:2d}  loss={tr["loss"]:.4f}  val_iou={va["iou"]:.4f} ✓')
    elif epoch % 10 == 0:
        print(f'P2 E{epoch:2d}  loss={tr["loss"]:.4f}  val_iou={va["iou"]:.4f}')

print(f'P2 done. Best: {best_iou:.4f}')
backup_to_drive()

## Phase 3: High-res @ 512px

In [ ]:
ckpt = torch.load(CKPT_DIR / 'best.pth', map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded P2 ({ckpt["val_iou"]:.4f})')
print(f'P3: 512px  |  30ep  |  lr=3e-5')

train_loader, val_loader, test_loader, _ = make_loaders(train_p, val_p, test_p, 512, 8, eval_batch=4)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-7)
scaler = GradScaler('cuda')

best_iou = 0.0
torch.cuda.empty_cache()

for epoch in range(1, 31):
    tr = run_epoch(model, train_loader, optimizer, scaler, True)
    va = run_epoch(model, val_loader, optimizer, scaler, False)
    scheduler.step()
    for k, v in [('train_loss', tr['loss']), ('train_iou', tr['iou']), ('val_loss', va['loss']), ('val_iou', va['iou'])]:
        history[k].append(v)
    if va['iou'] > best_iou:
        best_iou = va['iou']
        torch.save({'model_state': model.state_dict(), 'val_iou': best_iou}, CKPT_DIR / 'best.pth')
        if epoch % 10 == 0:
            print(f'P3 E{epoch:2d}  loss={tr["loss"]:.4f}  val_iou={va["iou"]:.4f} ✓')
    elif epoch % 10 == 0:
        print(f'P3 E{epoch:2d}  loss={tr["loss"]:.4f}  val_iou={va["iou"]:.4f}')

print(f'P3 done. Best: {best_iou:.4f}')
backup_to_drive()

## Test

In [ ]:
ckpt = torch.load(CKPT_DIR / 'best.pth', map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

_, _, test_loader, _ = make_loaders(train_p, val_p, test_p, 512, 4, eval_batch=4)

test_iou, test_dice = [], []
with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc='Test'):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        pred = model(imgs)
        m = compute_metrics(pred, masks, 0.5)
        test_iou.append(m['iou'])
        test_dice.append(m['dice'])

print(f'\nTest Results:')
print(f'  Crack IoU: {np.mean(test_iou):.4f}')
print(f'  Dice:      {np.mean(test_dice):.4f}')